# DNN Predictions — Format Conversion

Converts `dnn_predictions.csv` to match the format of `rf_predictions.csv`:

| Column | Source |
|---|---|
| `permno` | dnn_predictions |
| `date` | dnn_predictions |
| `Y` | features.csv (actual label) |
| `prob_DNN` | dnn_predictions (renamed from prob_outperform) |
| `period` | dnn_predictions |
| `test_gini` | computed per-period from test predictions (2×AUC−1) |

> **Note on Gini**: the RF file stores *training-set* Gini because the RF script computes it in-sample.  
> The DNN script does not save training predictions, so we compute the **test-set Gini** per period as a proxy (same formula: 2×AUC−1).

In [1]:
import pandas as pd
import numpy as np
from sklearn.metrics import roc_auc_score

DATA_DIR = "../data"
OUTPUT   = f"{DATA_DIR}/dnn_predictions_formatted.csv"

## 1. Load data

In [2]:
dnn  = pd.read_csv(f"{DATA_DIR}/dnn_predictions.csv", parse_dates=["date"])
feat = pd.read_csv(f"{DATA_DIR}/features.csv",        parse_dates=["date"])

print("DNN predictions:", dnn.shape)
print(dnn.dtypes)
dnn.head(3)

DNN predictions: (3833616, 4)
period                      int64
date               datetime64[ns]
permno                      int64
prob_outperform           float64
dtype: object


,period,date,permno,prob_outperform
0,1,1992-12-16,10104,0.546016
1,1,1992-12-16,10145,0.477352
2,1,1992-12-16,10161,0.525384


In [3]:
print("Features:", feat.shape)
feat[["permno", "date", "Y"]].head(3)

Features: (4116406, 34)


,permno,date,Y
0,10057,1990-12-11,0
1,10104,1990-12-11,1
2,10145,1990-12-11,1


## 2. Merge actual label Y

In [4]:
y_lookup = feat[["permno", "date", "Y"]].drop_duplicates()

merged = dnn.merge(y_lookup, on=["permno", "date"], how="left")

missing_y = merged["Y"].isna().sum()
print(f"Rows missing Y: {missing_y} / {len(merged)}")
merged.head(3)

Rows missing Y: 0 / 3833616


,period,date,permno,prob_outperform,Y
0,1,1992-12-16,10104,0.546016,0
1,1,1992-12-16,10145,0.477352,0
2,1,1992-12-16,10161,0.525384,0


## 3. Compute per-period test-set Gini  (2 × AUC − 1)

In [5]:
def period_gini(grp):
    y_true = grp["Y"]
    y_score = grp["prob_outperform"]
    if y_true.nunique() < 2:
        return np.nan
    auc = roc_auc_score(y_true, y_score)
    return 2 * auc - 1

gini_per_period = (
    merged.groupby("period")
          .apply(period_gini, include_groups=False)
          .rename("test_gini")
          .reset_index()
)

print(gini_per_period.to_string())

    period  test_gini
0        1   0.046562
1        2   0.049877
2        3   0.049148
3        4   0.039689
4        5   0.057804
5        6   0.030660
6        7   0.035075
7        8   0.043285
8        9   0.033875
9       10   0.036131
10      11   0.014844
11      12   0.006341
12      13  -0.000119
13      14   0.022741
14      15   0.031118
15      16   0.040274
16      17   0.009659
17      18   0.003065
18      19   0.011095
19      20   0.024086
20      21   0.002426
21      22   0.019667
22      23   0.001168
23      24   0.010502
24      25   0.010326
25      26   0.014399
26      27   0.024423
27      28   0.010870
28      29   0.010184
29      30  -0.003202
30      31   0.001012
31      32   0.011711


In [6]:
merged = merged.merge(gini_per_period, on="period", how="left")
merged.head(3)

,period,date,permno,prob_outperform,Y,test_gini
0,1,1992-12-16,10104,0.546016,0,0.046562
1,1,1992-12-16,10145,0.477352,0,0.046562
2,1,1992-12-16,10161,0.525384,0,0.046562


## 4. Rename & reorder to match RF format

In [7]:
out = (
    merged
    .rename(columns={"prob_outperform": "prob_DNN"})
    [["permno", "date", "Y", "prob_DNN", "period", "test_gini"]]
    .sort_values(["period", "date", "permno"])
    .reset_index(drop=True)
)

out["date"] = out["date"].dt.strftime("%Y-%m-%d")
out["Y"]    = out["Y"].astype("Int64")   # nullable int to preserve NaN if any

print(out.shape)
out.head(5)

(3833616, 6)


,permno,date,Y,prob_DNN,period,test_gini
0,10104,1992-12-16,0,0.546016,1,0.046562
1,10145,1992-12-16,0,0.477352,1,0.046562
2,10161,1992-12-16,0,0.525384,1,0.046562
3,10225,1992-12-16,0,0.490725,1,0.046562
4,10364,1992-12-16,0,0.493544,1,0.046562


## 5. Sanity check vs RF format

In [8]:
rf = pd.read_csv(f"{DATA_DIR}/rf_predictions.csv")

print("RF  columns:", rf.columns.tolist())
print("DNN columns:", out.columns.tolist())
print()
print("RF  shape:",  rf.shape)
print("DNN shape:",  out.shape)
print()
print("RF  prob range: ", rf["prob_RAF"].min(),   "–", rf["prob_RAF"].max())
print("DNN prob range: ", out["prob_DNN"].min(),  "–", out["prob_DNN"].max())
print()
print("RF  Gini range:", rf["train_gini"].min(),  "–", rf["train_gini"].max())
print("DNN Gini range:", out["test_gini"].min(),  "–", out["test_gini"].max())

RF  columns: ['permno', 'date', 'Y', 'prob_RAF', 'period', 'train_gini']
DNN columns: ['permno', 'date', 'Y', 'prob_DNN', 'period', 'test_gini']

RF  shape: (3832882, 6)
DNN shape: (3833616, 6)

RF  prob range:  0.2048469799553904 – 0.6464443444113165
DNN prob range:  0.016615447 – 0.93775976

RF  Gini range: 0.4175124456560923 – 0.5262043220986465
DNN Gini range: -0.0032017784070678124 – 0.057803520355277804


## 6. Save

In [9]:
out.to_csv(OUTPUT, index=False)
print(f"Saved {len(out):,} rows → {OUTPUT}")

Saved 3,833,616 rows → ../data/dnn_predictions_formatted.csv
